In [1]:
from segment_anything import SamAutomaticMaskGenerator, sam_model_registry
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import supervision as sv
import numpy as np
import torch
# import cv2
import numpy as np
import sys
import os
from tqdm import tqdm
from load_llff import load_llff_data
from img2vec_pytorch import Img2Vec

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
sam = sam_model_registry["default"](checkpoint="model/sam_vit_h_4b8939.pth")
sam.to(device)
mask_generator = SamAutomaticMaskGenerator(sam, pred_iou_thresh=0.80)

In [ ]:
imgs, poses, bds, render_poses, i_test = load_llff_data("data/nerf_llff_data/orchids", factor=4)
hwf = poses[0,:3,-1]
H, W, focal = hwf[0], hwf[1], hwf[2]
poses = poses[:,:3,:4]
hwf.shape, poses.shape # (3,) (N, 3, 4)
images = imgs
#convert images to uint8 ranging from 0 to 255
images = (images * 255).astype(np.uint8)

In [ ]:
masks = []
for image in tqdm(images):
    mask = mask_generator.generate(image)
    masks.append(mask)

In [ ]:
def delete_masks(masks, min_num_pixels):
    """delete the masks with less than min_num_pixels"""
    masks = [mask for mask in masks if np.sum(mask['segmentation']) > min_num_pixels]
    return masks
    
min_pixels = np.sqrt(H * W)
for i in range(len(masks)):
    masks[i] = delete_masks(masks[i], min_pixels)

In [ ]:
#Crop the images by their masks and store them in a N x M array, where N is the number of images and M is the number of maximum masks detected in the images. Add padding where there is no more masks.
max_masks = int(max([len(mask) for mask in masks]))
cropped_images = []
cropped_images_to_images = {}
for i in tqdm(range(len(images)), desc="Cropping images"):
    for j in range(len(masks[i])):
        mask = np.dstack([masks[i][j]['segmentation']]*3)
        cropped_images.append(Image.fromarray(images[i]*mask))
        cropped_images_to_images[len(cropped_images)-1] = i

In [ ]:
#downsample the cropped images by 4
downsampled_cropped_images = []
for cropped_image in tqdm(cropped_images, desc="Downsampling images"):
    downsampled_cropped_images.append(cropped_image.resize((cropped_image.width//4, cropped_image.height//4)))


In [ ]:
#find mask embeddings
img2vec = Img2Vec(cuda=True, model='efficientnet_b0')
# Define batch size
batch_size = 16
# Initialize an empty list to hold all image vectors
img_vectors = []

# Process images in batches
for i in tqdm(range(0, len(downsampled_cropped_images), batch_size), desc="Converting to embeddings"):
    batch = downsampled_cropped_images[i:i+batch_size]
    batch_vectors = img2vec.get_vec(batch)
    img_vectors.extend(batch_vectors)

# Convert list to numpy array
img_vectors = np.array(img_vectors)

In [ ]:
def build_cannot_link_constraints(img_vectors, cropped_images_to_images):
    cannot_link_constraints = []
    for i in range(len(img_vectors)):
        for j in range(i+1, len(img_vectors)):
            if cropped_images_to_images[i] == cropped_images_to_images[j]:
                cannot_link_constraints.append((i, j))
    return cannot_link_constraints

In [ ]:
from sklearn.metrics.pairwise import cosine_distances
import hdbscan

# Calculate the cosine distance matrix
distance_matrix = cosine_distances(img_vectors).astype(np.float64)
# List of cannot-link pairs (indices of points that shouldn't be in the same cluster)
cannot_link_constraints = build_cannot_link_constraints(img_vectors, cropped_images_to_images)

for (i, j) in cannot_link_constraints:
    distance_matrix[i, j] = 1.0
    distance_matrix[j, i] = 1.0

In [ ]:
# #plot number of outliers against min_cluster_size
# min_cluster_sizes = range(3,len(images))
# outliers = []
# clusters = []
# for min_cluster_size in tqdm(min_cluster_sizes, desc="Finding optimal min_cluster_size"):
#     clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=None, metric='precomputed',cluster_selection_method='eom')
#     cluster_labels = clusterer.fit_predict(distance_matrix)
#     outliers.append(np.sum(cluster_labels == -1))
#     clusters.append(len(set(cluster_labels)))
# plt.plot(min_cluster_sizes, outliers)
# plt.xlabel("min_cluster_size")
# plt.ylabel("Number of outliers")
# plt.show()
# plt.plot(min_cluster_sizes, clusters)
# plt.xlabel("min_cluster_size")
# plt.ylabel("Number of clusters")
# plt.show()

In [ ]:
# #plot outliers and clusters against min_samples in the same plot
# min_cluster_sizes = range(3,len(images))
# #standardize the outliers and clusters by z score
# outliers = (np.array(outliers) - np.mean(outliers))/ np.std(outliers)
# clusters = (np.array(clusters) - np.mean(clusters))/ np.std(clusters)
# plt.plot(min_cluster_sizes, outliers, label="Number of outliers")
# plt.plot(min_cluster_sizes, clusters, label="Number of clusters")
# plt.xlabel("min_cluster_size")
# plt.legend()
# plt.show()

In [ ]:
# #at which index do the outliers and the clusters intersect?
# intersection = np.argwhere(np.diff(np.sign(outliers - clusters))).flatten()
# min_cluster_size = intersection[0]
# for i in range(1, len(intersection)):
#     if outliers[intersection[i]] < min_cluster_size:
#         min_cluster_size = intersection[i]
    
# print(min_cluster_size, outliers[min_cluster_size])

In [ ]:
# Create an HDBSCAN object with cosine distance
clusterer = hdbscan.HDBSCAN(min_cluster_size=len(images)//4, 
                            min_samples=None, 
                            metric='precomputed',
                            cluster_selection_method='eom') #cluster_selection_epsilon=0.01

# Fit the model using the modified distance matrix
cluster_labels = clusterer.fit_predict(distance_matrix)

# Output the cluster labels
print(cluster_labels)
plt.hist(cluster_labels, bins=len(set(cluster_labels)))
plt.show()
print("Number of clusters: ", len(set(cluster_labels)))
print("Number of outliers: ", np.sum(cluster_labels == -1))


In [ ]:
#display the images in the same cluster, randomly select 5 clusters and display 4 images from each cluster
import random
import matplotlib.pyplot as plt

def display_images(images, cluster_labels, n_clusters, n_images_per_cluster):
    fig, axs = plt.subplots(n_clusters, n_images_per_cluster, figsize=(22,10))
    for i in range(n_clusters):
        cluster = random.choice(list(set(cluster_labels)))
        cluster_indices = [index for index, label in enumerate(cluster_labels) if label == cluster]
        for j in range(n_images_per_cluster):
            img = images[random.choice(cluster_indices)]
            axs[i, j].imshow(img)
            axs[i, j].axis('off')
    plt.show()

display_images(cropped_images, cluster_labels, 5, 13)


In [ ]:
#display some images from the outlier cluster, i.e. the cluster with label -1
outlier_cluster_indices = [index for index, label in enumerate(cluster_labels) if label == -1]
outlier_images = [cropped_images[index] for index in outlier_cluster_indices]
fig, axs = plt.subplots(2, 10, figsize=(20,5))
for i in range(20):
    img = random.choice(outlier_images)
    axs[i//10, i%10].imshow(img)
plt.show()

In [ ]:
# from sklearn.manifold import TSNE
# import matplotlib.pyplot as plt

# # Applying t-SNE to reduce dimensions for visualization
# tsne = TSNE(metric="precomputed")
# transformed_data = tsne.fit_transform(distance_matrix)

# # Plotting
# plt.scatter(transformed_data[:, 0], transformed_data[:, 1], c=cluster_labels, cmap='viridis', alpha=0.5)
# plt.colorbar()
# plt.title('t-SNE visualization of the clustering')
# plt.show()


In [ ]:
#save the img_vectors as tsv file
# import pandas as pd
# df = pd.DataFrame(img_vectors)
# df.to_csv("img_vectors.tsv", sep="\t", header=False, index=False)

In [ ]:
def cropped_image_to_pixel_ids(cropped_image):
    # Convert to numpy and take the 0th channel
    cropped_image = np.array(cropped_image)[:,:,0]

    # Get the indices of the pixels that are greater than 0
    i, j = np.where(cropped_image > 0)

    # Convert the indices to pixel IDs
    pixel_ids = i * cropped_image.shape[1] + j

    return pixel_ids.tolist()

In [ ]:
#create a graph with image ids as nodes
import networkx as nx
G = nx.Graph()
for i in range(len(images)):
    G.add_node(i, type="image")

#add mask nodes
for i in range(len(cropped_images)):
    G.add_node(f"{cropped_images_to_images[i]}.{i}", type="mask")

#add concept nodes
for i in range(len(cluster_labels)):
    #do not add outliers as concepts
    if cluster_labels[i] != -1:
        G.add_node(f"concept_{cluster_labels[i]}", type="concept")
    
#connect the cropped images to their original images in the graph
for i in range(len(cropped_images)):
    G.add_edge(f"{cropped_images_to_images[i]}.{i}", cropped_images_to_images[i], type = "has_mask")

print("Added edges between cropped images and their original images")
#connect cropped_images that are in the same cluster
for i in range(len(cropped_images)):
    for j in range(i+1, len(cropped_images)):
        if cluster_labels[i] == cluster_labels[j] and cropped_images_to_images[i] != cropped_images_to_images[j]:
            G.add_edge(f"{cropped_images_to_images[i]}.{i}", f"{cropped_images_to_images[j]}.{j}", type="same_concept")

print("Added edges between cropped images that are in the same cluster")
#connect cropped_images to the cluster_ids they belong to
for i in range(len(cropped_images)):
    if cluster_labels[i] != -1:
        G.add_edge(f"concept_{cluster_labels[i]}", f"{cropped_images_to_images[i]}.{i}", type="has_concept")
print("Added edges between cropped images and the cluster_ids they belong to")
#connect images to cluster_ids their cropped images belong to
for i in range(len(cropped_images)):
    if cluster_labels[i] != -1:
        G.add_edge(cropped_images_to_images[i], f"concept_{cluster_labels[i]}", type="has_concept")
print("Added edges between images and the cluster_ids their cropped images belong to")
#connect mask pixels to the cropped images they belong to
for i in tqdm(range(len(cropped_images)), desc="Adding pixel bags"):
    if cluster_labels[i] == -1:
        continue
    pixel_ids = cropped_image_to_pixel_ids(cropped_images[i])
    #add pixel_bag node
    G.add_node(f"pixel_bag_{i}", type="pixel_bag", value = pixel_ids)
    G.add_edge(f"{cropped_images_to_images[i]}.{i}", f"pixel_bag_{i}", type="has_pixel_bag")
    G.add_edge(cropped_images_to_images[i], f"pixel_bag_{i}", type="has_pixel_bag")
    G.add_edge(f"concept_{cluster_labels[i]}", f"pixel_bag_{i}", type="has_pixel_bag")
print("Added edges between mask pixels and the cropped images they belong to")
    

In [ ]:
from collections import Counter

node_types = nx.get_node_attributes(G, 'type')
type_counts = Counter(node_types.values())

for node_type, count in type_counts.items():
    print(f"Type: {node_type}, Count: {count}")

#display edges between mask nodes
mask_edges = [(u, v) for u, v in G.edges if node_types.get(u) == 'mask' and node_types.get(v) == 'mask']
print("Number of mask edges: ", len(mask_edges))

In [ ]:
#recursively combine masks which are connected into single node called "concept", until no more masks remain
# def combine_masks(G):
#     node_types = nx.get_node_attributes(G, 'type')
#     mask_edges = [(u, v) for u, v in G.edges if node_types[u] == 'mask' and node_types[v] == 'mask']
#     print("Number of mask edges: ", len(mask_edges))
#     while len(mask_edges) > 0:
#         u, v = mask_edges[0]
#         #combine the masks
#         concept = f"{u}-{v}"
#         G.add_node(concept, type='concept')
#         #add edges between the neighbors of u and v to the concept node
#         u_neighbors = list(G.neighbors(u))
#         v_neighbors = list(G.neighbors(v))
#         for neighbor in u_neighbors:
#             G.add_edge(concept, neighbor)
#         for neighbor in v_neighbors:
#             G.add_edge(concept, neighbor)
#         #remove the mask nodes u and v from the graph
#         G.remove_node(u)
#         G.remove_node(v)
#         #add the new concept node to the graph
#         node_types = nx.get_node_attributes(G, 'type')
#         mask_edges = [(u, v) for u, v in G.edges if 
#                       (node_types[u] == 'mask' or node_types[u] =='concept') and (node_types[v] == 'mask' or node_types[v] == 'concept')]
    
#     #rename the remaining mask nodes to concept nodes
#     node_types = nx.get_node_attributes(G, 'type')
#     mask_nodes = [node for node in G.nodes if node_types[node] == 'mask']
#     for node in mask_nodes:
#         G.nodes[node]['type'] = 'concept'
#     return G

# G = combine_masks(G)
# node_types = nx.get_node_attributes(G, 'type')
# type_counts = Counter(node_types.values())

# for node_type, count in type_counts.items():
#     print(f"Type: {node_type}, Count: {count}")

In [ ]:
#greedily select k images that selects the concepts with the most unique pixels
def calculate_pixel_contribution(current_selection, candidate_image, G):
    # Calculate the contribution of candidate_image when added to current_selection
    new_pixels = set()
    for concept in G.neighbors(candidate_image):
        if G.nodes[concept]['type'] == 'concept':
            current_pixels = set()
            # Union pixels from already selected images for this concept
            for img in current_selection:
                for bag_node in G.neighbors(img):
                    if G.nodes[bag_node]['type'] == 'pixel_bag' and bag_node in G.neighbors(concept):
                        current_pixels.update(G.nodes[bag_node]['value'])

            # Pixels from the candidate image for this concept
            candidate_pixels = set()
            for bag_node in G.neighbors(candidate_image):
                if G.nodes[bag_node]['type'] == 'pixel_bag' and bag_node in G.neighbors(concept):
                    candidate_pixels.update(G.nodes[bag_node]['value'])
            
            # New pixels contributed by candidate image
            new_pixels.update(candidate_pixels - current_pixels)
    return len(new_pixels)

def greedy_select_images(G, k):
    all_images = [node for node in G.nodes if G.nodes[node]['type'] == 'image']
    selected_images = []
    remaining_images = set(all_images)

    while len(selected_images) < k and remaining_images:
        max_contribution = 0
        best_image = None
        
        # Evaluate each remaining image's contribution
        for image in tqdm(remaining_images, desc="Evaluating images"):
            contribution = calculate_pixel_contribution(selected_images, image, G)
            if contribution > max_contribution:
                max_contribution = contribution
                best_image = image

        if best_image != None:
            selected_images.append(best_image)
            remaining_images.remove(best_image)
        else:
            break  # No more images contribute new pixels

    return selected_images

# Assume G is already created and populated with image-concept-pixel data
# Example: k = 5
selected_images = greedy_select_images(G, 25)
print("Selected images:", selected_images)

In [ ]:
#display the selected images
fig, axs = plt.subplots(4,5, figsize=(20,20))
for i in range(20):
    img = images[selected_images[i]]
    axs[i//5, i%5].imshow(img)
    axs[i//5, i%5].axis('off')

In [ ]:
#plot the camera positions of only the selected images in 3D
camera_positions = poses[:, :3, 3]
selected_camera_positions = np.array([camera_positions[i] for i in selected_images[:20]])
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(selected_camera_positions[:, 0], selected_camera_positions[:, 1], selected_camera_positions[:, 2])
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.show()

In [ ]:
#plot the camera positions of only the selected images in 3D
camera_positions = poses[:, :3, 3]
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(camera_positions[:, 0], camera_positions[:, 1], camera_positions[:, 2])
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.show()